# 01 — Validación de instancias

Visualización de cobertura geográfica para cada instancia generada.
Identifica edificios sin cobertura y candidatos del borde.

In [10]:
import sys
sys.path.insert(0, "../src/python")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from instancia import load_instance
from lagrangiana import precompute_valid_candidates

def analyze_instance(radius):
    """Carga instancia y calcula cobertura."""
    path = f"../data/processed/instancia_laguna_{radius}m.json"
    inst = load_instance(path)
    vc = precompute_valid_candidates(inst)
    
    n_i, n_k = len(inst.I), len(inst.K)
    tipo_nombres = {0: "Orgánica", 1: "Resto", 2: "Reciclable", 3: "Peligrosos"}
    
    # Clasificar edificios
    uncovered_buildings = {}  # i → lista de tipos sin cobertura
    for i in range(n_i):
        missing = [k for k in range(n_k) if not vc[i][k]]
        if missing:
            uncovered_buildings[i] = missing
    
    # Resumen
    print(f"{'═' * 50}")
    print(f"  Instancia {radius}m")
    print(f"{'═' * 50}")
    print(f"  Edificios:    {n_i}")
    print(f"  Candidatos:   {len(inst.J)}")
    print(f"  Sin cobertura: {len(uncovered_buildings)} edificios")
    if uncovered_buildings:
        for i, tipos in uncovered_buildings.items():
            nombres = [tipo_nombres[k] for k in tipos]
            print(f"    edificio {i}: falta {nombres}")
    print()
    
    return inst, vc, uncovered_buildings

In [11]:
def plot_coverage_map(inst, vc, uncovered_buildings, radius):
    """Mapa de cobertura: edificios cubiertos vs no cubiertos."""
    n_i = len(inst.I)
    
    # Coordenadas
    bld_lon = np.array([inst.I[i].longitude for i in range(n_i)])
    bld_lat = np.array([inst.I[i].latitude for i in range(n_i)])
    cand_lon = np.array([inst.J[j].longitude for j in range(len(inst.J))])
    cand_lat = np.array([inst.J[j].latitude for j in range(len(inst.J))])
    
    # Separar edificios cubiertos y no cubiertos
    covered_idx = [i for i in range(n_i) if i not in uncovered_buildings]
    uncovered_idx = list(uncovered_buildings.keys())
    
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # Edificios cubiertos
    ax.scatter(
        bld_lon[covered_idx], bld_lat[covered_idx],
        s=10, c="steelblue", alpha=0.5, zorder=3, label=f"Cubiertos ({len(covered_idx)})"
    )
    
    # Edificios SIN cobertura
    if uncovered_idx:
        ax.scatter(
            bld_lon[uncovered_idx], bld_lat[uncovered_idx],
            s=60, c="red", marker="X", edgecolors="darkred", linewidth=1,
            zorder=5, label=f"Sin cobertura ({len(uncovered_idx)})"
        )
        # Etiquetar cada uno
        for i in uncovered_idx:
            tipos = uncovered_buildings[i]
            tipo_str = ",".join(str(k) for k in tipos)
            ax.annotate(
                f"k={tipo_str}",
                (bld_lon[i], bld_lat[i]),
                textcoords="offset points", xytext=(8, 8),
                fontsize=7, color="darkred", fontweight="bold"
            )
    
    # Candidatos
    ax.scatter(
        cand_lon, cand_lat,
        s=25, c="limegreen", marker="^", edgecolors="darkgreen",
        linewidth=0.5, zorder=4, label=f"Candidatos ({len(inst.J)})"
    )
    
    ax.set_title(
        f"Cobertura — Instancia {radius}m  |  "
        f"{n_i} edificios, {len(inst.J)} candidatos, "
        f"{len(uncovered_idx)} sin cobertura",
        fontsize=13, fontweight="bold"
    )
    ax.set_xlabel("Longitud")
    ax.set_ylabel("Latitud")
    ax.legend(loc="upper left")
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

In [12]:
for r in [250, 400, 450, 500, 550, 700, 750, 800, 1000, 1500]:
    try:
        inst, vc, uncovered = analyze_instance(r)
        if uncovered:
            plot_coverage_map(inst, vc, uncovered, r)
        else:
            print(f"  ✅ Cobertura completa — no se genera mapa\n")
    except FileNotFoundError:
        print(f"  ⚠️ Archivo no encontrado para {r}m\n")

══════════════════════════════════════════════════
  Instancia 250m
══════════════════════════════════════════════════
  Edificios:    243
  Candidatos:   76
  Sin cobertura: 0 edificios

  ✅ Cobertura completa — no se genera mapa

══════════════════════════════════════════════════
  Instancia 400m
══════════════════════════════════════════════════
  Edificios:    546
  Candidatos:   133
  Sin cobertura: 0 edificios

  ✅ Cobertura completa — no se genera mapa

══════════════════════════════════════════════════
  Instancia 450m
══════════════════════════════════════════════════
  Edificios:    640
  Candidatos:   153
  Sin cobertura: 0 edificios

  ✅ Cobertura completa — no se genera mapa

══════════════════════════════════════════════════
  Instancia 500m
══════════════════════════════════════════════════
  Edificios:    740
  Candidatos:   172
  Sin cobertura: 0 edificios

  ✅ Cobertura completa — no se genera mapa

══════════════════════════════════════════════════
  Instancia 550m
═